[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/arap/arap.ipynb)

# As Rigid As Possible

This example computes as-rigid-as-possible deformation of a triangulated surface as one end of a square bar stays clamped while the opposite end bends upward through a quarter turn. Every free vertex moves so that its incident edge star stays close to a rigid rotation of its rest shape, allowing the bar to curve smoothly without shearing or collapsing its cross-section. Alternating between fitting a local rotation at each vertex and solving for vertex positions that match those rotated edges relaxes the surface toward minimal distortion.

At each vertex, aligning rest edges with deformed edges defines the symmetric bilinear form `edges.dot(Rotor >> rest_edges)` on rotors; pulling those edge forms back onto vertices yields every optimal vertex rotor at once as the top eigenvector of one batched `.eigh()` call, with no singular value decompositions or reflection sign fixes. Averaging the linear rotation maps `rotors >> Point` across each edge's endpoints stays invariant to eigenvector sign flips, and pulling edge differences back across `d[0].adjugate()` forms the cotangent Laplacian solved in the local-global iteration of [Sorkine and Alexa (2007)](https://igl.ethz.ch/projects/ARAP/arap_web.pdf).

In matrix geometry-processing notation, minimizing $E = \sum_i \sum_{j \in N(i)} w_{ij} \|(\mathbf p'_i - \mathbf p'_j) - R_i (\mathbf p_i - \mathbf p_j)\|^2$ assembles a $3 \times 3$ cross-covariance matrix $S_i = \sum_{j \in N(i)} w_{ij}\, \mathbf e_{ij} {\mathbf e'_{ij}}^{\mathsf T}$ at each vertex, computes its singular value decomposition $S_i = U_i \Sigma_i V_i^{\mathsf T}$, inserts $\operatorname{diag}(1, 1, \det(V_i U_i^{\mathsf T}))$ so $R_i = V_i \operatorname{diag}(1, 1, \det(V_i U_i^{\mathsf T})) U_i^{\mathsf T}$ stays in $\mathrm{SO}(3)$ rather than reflecting, and solves three scalar Poisson systems against the cotangent matrix $L \mathbf P' = \mathbf b$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.surfaces.arap import render, scenarios

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Rotor = ga.gatype.rotor()                           # Rotor+

## Square bar and handle trajectory

The surface is a closed square tube stretched from a subdivided cube and textured with a rest-pose checkerboard. Vertices on the near end stay clamped in place while vertices on the far end follow a circular arc of the bar's length, turning upward in the vertical plane across the sequence of handle poses.

In [ ]:
n_divisions = scenarios.n_divisions
n_frames = scenarios.n_frames
tile_size = scenarios.tile_size
bar_topology, rest_vertices, handles, poses = scenarios.bending_setup(n_divisions, n_frames)
save_figure(render.bar_figure(bar_topology, rest_vertices, [rest_vertices, poses[-1]], tile_size), "arap_handles");

The left view shows the straight bar at rest with square checkerboard tiles, and the right view shows the final target configuration with only the far-end cap carried upward and turned vertical.

## Cotangent Laplacian editing

Applying `d[0]` to the vertex field forms the oriented edge vectors, and each edge's cotangent weight averages the cotangents of the two triangle corners facing that edge. Pulling weighted edge differences back to vertices via `d[0].adjugate()` and adding a stiff handle penalty forms the pinned cotangent Laplacian; matching the unrotated rest edges bends the bar toward the handle target, but shears and crushes its cross-section.

In [ ]:
stiffness = scenarios.stiffness

In [ ]:
def cotangent_weights(topology: Topology, rest_edges: Point) -> Scalar:
    """Half the cotangents of the two corners opposite each edge form the edge cotangent weights (`Scalar+[E]`)."""
    triangle_edges = topology.boundary[2, 1] * rest_edges                       # [3] Point-[F]
    tails, heads = topology.corners[2, 1]
    next_edges, prev_edges = triangle_edges[tails], triangle_edges[heads]       # [3] Point-[F], [3] Point-[F]
    cotangents = -(next_edges.dual_inverse() & prev_edges) / next_edges.commutator(prev_edges).norm()  # [3] Scalar+[F]
    return (topology.incidence[2, 1].adjugate() * (cotangents / 2)).sum(axis=0)  # Scalar+[E]


def corner_cosines(topology: Topology, vertices: Point) -> Scalar:
    """Normalized inner products of the two directed triangle sides meeting at each corner give the corner angle cosines (`[3] Scalar+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                # [3] Point-[F]
    tails, heads = topology.corners[2, 1]
    next_edges, prev_edges = sides[tails], sides[heads]                         # [3] Point-[F], [3] Point-[F]
    return -(next_edges.dual_inverse() & prev_edges) / (next_edges.antinorm() * prev_edges.antinorm())  # [3] Scalar+[F]


edge_diffs = bar_topology.d[0]                                                  # Scalar+[E <- V]
edge_means = bar_topology.incidence[1, 0].mean(axis=0)                          # Scalar+[E <- V]
rest_edges = edge_diffs * rest_vertices                                         # Point-[E]
edge_weights = spdiag(cotangent_weights(bar_topology, rest_edges))              # Scalar+[E <- E]
handle_penalty = spdiag(mv.scalar((handles * stiffness)[..., None]).field())    # Scalar+[V <- V]
handle_laplacian = (edge_diffs.adjugate() * edge_weights * edge_diffs + handle_penalty) * Point  # (Point- <- Point-)[V <- V]

laplacian_end = handle_laplacian.solve(edge_diffs.adjugate() * edge_weights * rest_edges + handle_penalty * poses[-1])  # Point-[V]
print((edge_diffs * Scalar).gatype, edge_diffs.shape)
print(handle_laplacian.gatype, handle_laplacian.shape)
save_figure(render.bar_figure(bar_topology, rest_vertices, [laplacian_end], tile_size), "arap_laplacian");

Because the target edge differences remain oriented along the horizontal rest bar while the far cap turns vertical, the Laplacian solution flattens and tapers the bent midsection.

## Local vertex rotors and global position update

For fixed vertex positions, each edge's alignment `edges.dot(Rotor >> rest_edges)` is a quadratic form on rotors that pulls back onto its two endpoints; every even multivector in three-dimensional Euclidean space is a scaled rotor, so the top eigenvector of each vertex's summed form gives its optimal rotation directly. Sandwiching `Point` by `rotors` turns each rotor into a linear rotation map `Point- <- Point-` whose sign cancels out, so averaging those endpoint maps across each edge and solving the same Laplacian system updates the surface without sign ambiguity.

In [ ]:
def local_rotors(edge_means, edge_weights, rest_edges: Point, edges: Point) -> Rotor:
    """Summing each edge's alignment quadratic form onto its endpoints yields the optimal vertex rotors (`Rotor+[V]`) as top eigenvectors."""
    forms = edge_means.adjugate() * edge_weights * edges.dot(Rotor >> rest_edges)  # (Scalar+ <- (Rotor+, Rotor+))[V]
    _, modes = forms.batch().eigh()                                             # [..., 4] Rotor+
    return modes[..., -1].field()                                               # Rotor+[V]


def global_vertices(edge_diffs, edge_means, edge_weights, handle_penalty, handle_laplacian,
                    rest_edges: Point, rotors: Rotor, targets: Point) -> Point:
    """Averaging the endpoint rotation maps across each edge and solving the handle-pinned cotangent Laplacian updates the deformed vertices (`Point-[V]`)."""
    turned_edges = (edge_means * (rotors >> Point))(rest_edges)                 # Point-[E]
    return handle_laplacian.solve(edge_diffs.adjugate() * edge_weights * turned_edges + handle_penalty * targets)  # Point-[V]


initial_edges = edge_diffs * laplacian_end                                      # Point-[E]
first_rotors = local_rotors(edge_means, edge_weights, rest_edges, initial_edges)  # Rotor+[V]
first_step_vertices = global_vertices(
    edge_diffs, edge_means, edge_weights, handle_penalty, handle_laplacian, rest_edges, first_rotors, poses[-1]
)                                                                               # Point-[V]
save_figure(render.bar_figure(bar_topology, rest_vertices, [laplacian_end, first_step_vertices], tile_size), "arap_one_step");

Comparing the pure Laplacian edit on the left against a single local-global pass on the right shows the bent midsection immediately recovering most of its square cross-section.

## Warm-started bending across the arc

Both local rotor fitting and the global Laplacian solve lower the same cotangent-weighted edge mismatch energy, so alternating them converges rapidly at each pose. Warm-starting each frame from the previous frame's deformed bar streams the full quarter-turn bend in a single pass.

In [ ]:
n_iterations = scenarios.n_iterations
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def deformation_states(
    topology: Topology,
    rest_vertices: Point,
    handles: np.ndarray,
    poses: Iterable[Point],
    stiffness: float,
    n_iterations: int,
) -> Iterator[tuple[Point, Point, Scalar]]:
    """Yields `(rigid_vertices, laplacian_vertices, energies)` for each handle pose across `n_iterations` warm-started local-global steps."""
    edge_diffs = topology.d[0]                                                  # Scalar+[E <- V]
    edge_means = topology.incidence[1, 0].mean(axis=0)                          # Scalar+[E <- V]
    rest_edges = edge_diffs * rest_vertices                                     # Point-[E]
    edge_weights = spdiag(cotangent_weights(topology, rest_edges))              # Scalar+[E <- E]
    handle_penalty = spdiag(mv.scalar((handles * stiffness)[..., None]).field())  # Scalar+[V <- V]
    handle_laplacian = (edge_diffs.adjugate() * edge_weights * edge_diffs + handle_penalty) * Point  # (Point- <- Point-)[V <- V]

    vertices = rest_vertices                                                    # Point-[V]
    for targets in poses:
        energies = []
        for _ in range(n_iterations):
            edges = edge_diffs * vertices                                       # Point-[E]
            rotors = local_rotors(edge_means, edge_weights, rest_edges, edges)  # Rotor+[V]
            turned_edges = (edge_means * (rotors >> Point))(rest_edges)         # Point-[E]
            # Because each rotor preserves edge length, the two endpoint mismatches sum to 2 * (edge^2 + rest^2 - 2 * edge . turned).
            mismatch = edge_weights * (edges.norm_squared() + rest_edges.norm_squared() - 2 * edges.dot(turned_edges))  # Scalar+[E]
            penalty = handle_penalty * (vertices - targets).norm_squared()      # Scalar+[V]
            energies.append(2 * mismatch.sites.sum() + penalty.sites.sum())
            vertices = handle_laplacian.solve(edge_diffs.adjugate() * edge_weights * turned_edges + handle_penalty * targets)  # Point-[V]
        laplacian_vertices = handle_laplacian.solve(edge_diffs.adjugate() * edge_weights * rest_edges + handle_penalty * targets)  # Point-[V]
        yield vertices, laplacian_vertices, stack(energies)


states = show_states(
    deformation_states(bar_topology, rest_vertices, handles, poses, stiffness, n_iterations),
    render.bar_animation, bar_topology, rest_vertices, tile_size,
    name="arap_bending", duration_ms=frame_duration_ms, n_frames=n_frames,
)
final_rigid, final_laplacian, last_energies = states[-1]
save_figure(render.bar_figure(bar_topology, rest_vertices, [final_rigid, final_laplacian], tile_size), "arap_comparison");

Across the full quarter-turn bend, the converged as-rigid-as-possible surface on the left preserves its square checkerboard tiles and uniform thickness, whereas the Laplacian edit on the right collapses inward along the bend.

In [ ]:
# checks
# Evaluating an edge's alignment quadratic form at a rotor matches the inner product with the rotated rest edge.
edge_forms = initial_edges.dot(Rotor >> rest_edges)                             # (Scalar+ <- (Rotor+, Rotor+))[E]
sample_rotor = first_rotors.batch()[0]                                          # [] Rotor+
assert ((edge_forms(sample_rotor, sample_rotor) - initial_edges.dot(sample_rotor >> rest_edges)).batch().abs() < 1e-12).all()
# Negating a rotor leaves its linear rotation map on points unchanged.
assert (((first_rotors >> Point)(rest_vertices) - (-first_rotors >> Point)(rest_vertices)).antinorm().batch() < 1e-12).all()
# At the final frame the energy decreases monotonically across iterations and the handles match their target pose.
assert ((last_energies[1:] - last_energies[:-1]) <= 1e-9 * last_energies[0]).all()
assert ((final_rigid - poses[-1]).antinorm().batch()[handles] < 1e-3).all()
# Corner angle cosines distort less than half as much as under Laplacian editing.
rest_cosines = corner_cosines(bar_topology, rest_vertices)                      # [3] Scalar+[F]
rigid_distortion = (corner_cosines(bar_topology, final_rigid) - rest_cosines).abs().sites.mean().mean()
laplacian_distortion = (corner_cosines(bar_topology, final_laplacian) - rest_cosines).abs().sites.mean().mean()
assert (rigid_distortion < 0.5 * laplacian_distortion).all()